# PARC2026 — Dataset Inventory V1

運営combined datasetまたは公開LIBERO-plusについて、**task × episode構成を最初に可視化する**Notebookです。

このNotebookは単体で実行できます。冒頭で `00_a100_preflight.ipynb` 相当のworkspace/repo準備を行い、運営datasetがColabに無い場合は公開 `Sylvest/libero_plus_lerobot` の **metadataだけ** を取得してInventory pipelineを先に検証します。公開datasetの結果と運営datasetの正本Inventoryは混同しません。

## Self-contained preflight
`00_a100_preflight.ipynb` を別に実行していなくても、このセルだけで同じworkspace準備を行います。

In [ ]:
from pathlib import Path
import json, os, platform, subprocess, sys
import pandas as pd

print('python:', sys.version)
print('platform:', platform.platform())
if subprocess.run(['bash', '-lc', 'command -v nvidia-smi'], capture_output=True).returncode == 0:
    subprocess.run(['nvidia-smi'], check=True)
else:
    print('nvidia-smi: not found (Inventory itself can run on CPU)')

ROOT = Path('/content/parc2026')
for p in [ROOT, ROOT/'vendor', ROOT/'cache', ROOT/'datasets', ROOT/'outputs']:
    p.mkdir(parents=True, exist_ok=True)
REPO = ROOT / 'py_AI'
if not (REPO / '.git').exists():
    subprocess.run(['git', 'clone', 'https://github.com/yu37330/py_AI.git', str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'fetch', '--all', '--prune'], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', 'main'], check=True)
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', 'main'], check=True)
print('workspace:', ROOT)
print('repo:', REPO)
print('git:', subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'], text=True).strip())

## Dataset rootを解決
優先順位は `PARC_DATASET_ROOT` → Colab上の運営 `libero_combined_20hz` → 公開LIBERO-plus metadata fallback です。

fallbackでは15GB級の動画は取得せず、Inventoryに必要な `meta/info.json` / `meta/episodes.jsonl` / `meta/tasks.jsonl` だけを取得します。

In [ ]:
PUBLIC_DATASET_ID = os.environ.get('PARC_PUBLIC_INVENTORY_DATASET', 'Sylvest/libero_plus_lerobot')
configured = os.environ.get('PARC_DATASET_ROOT')
organizer_root = ROOT / 'datasets' / 'libero_combined_20hz'

if configured:
    DATASET_ROOT = Path(configured)
    DATASET_SOURCE = 'configured'
elif (organizer_root / 'meta' / 'info.json').exists():
    DATASET_ROOT = organizer_root
    DATASET_SOURCE = 'organizer_combined'
else:
    PUBLIC_ROOT = ROOT / 'datasets' / 'public_libero_plus_metadata'
    if not (PUBLIC_ROOT / 'meta' / 'info.json').exists():
        try:
            from huggingface_hub import snapshot_download
        except ImportError:
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'huggingface_hub'], check=True)
            from huggingface_hub import snapshot_download
        print('organizer dataset not found; downloading public metadata only:', PUBLIC_DATASET_ID)
        snapshot_download(
            repo_id=PUBLIC_DATASET_ID,
            repo_type='dataset',
            local_dir=str(PUBLIC_ROOT),
            allow_patterns=['meta/info.json', 'meta/episodes.jsonl', 'meta/tasks.jsonl'],
        )
    DATASET_ROOT = PUBLIC_ROOT
    DATASET_SOURCE = f'public_metadata:{PUBLIC_DATASET_ID}'

if not (DATASET_ROOT / 'meta' / 'info.json').exists():
    raise FileNotFoundError(f'meta/info.json not found: {DATASET_ROOT}')
if not (DATASET_ROOT / 'meta' / 'episodes.jsonl').exists():
    raise FileNotFoundError(f'meta/episodes.jsonl not found: {DATASET_ROOT}')
print('dataset source:', DATASET_SOURCE)
print('dataset root:', DATASET_ROOT)
if DATASET_SOURCE.startswith('public_metadata:'):
    print('NOTE: これは公開LIBERO-plusのInventoryです。運営19,533 episodeの正本ではありません。')

## Metadataを確認
fps、features、episode/frame数の宣言値を確認します。

In [ ]:
info = json.loads((DATASET_ROOT / 'meta' / 'info.json').read_text())
print(json.dumps(info, ensure_ascii=False, indent=2)[:12000])

## Inventoryを生成
metadataのみを読み、episode/task Inventoryとsummaryを生成します。

In [ ]:
OUT = ROOT / 'outputs' / 'dataset_inventory_v1'
OUT.mkdir(parents=True, exist_ok=True)
subprocess.run([
    sys.executable,
    str(REPO / 'tools/data/build_dataset_inventory.py'),
    '--root', str(DATASET_ROOT),
    '--out', str(OUT),
], check=True)
print('outputs:', OUT)

In [ ]:
summary = json.loads((OUT / 'dataset_inventory_summary.json').read_text())
summary['inventory_source'] = DATASET_SOURCE
(OUT / 'dataset_inventory_summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2) + '\n')
episodes = pd.read_csv(OUT / 'episode_inventory.csv')
tasks = pd.read_csv(OUT / 'task_inventory.csv')
display(pd.DataFrame([summary]))
display(tasks.head(50))

## Task imbalance候補
Raw / Uniform / Sqrt-balanced の3案を作り、次のcheap ablation候補にします。

In [ ]:
import numpy as np
sampling = tasks[['task_name', 'episodes']].copy()
sampling['raw_prob'] = sampling['episodes'] / sampling['episodes'].sum()
sampling['uniform_prob'] = 1.0 / len(sampling)
sqrt_n = np.sqrt(sampling['episodes'].clip(lower=1))
sampling['sqrt_balanced_prob'] = sqrt_n / sqrt_n.sum()
sampling = sampling.sort_values('episodes', ascending=False)
sampling.to_csv(OUT / 'task_sampling_candidates.csv', index=False)
display(sampling.head(50))
print('max/min episode ratio:', sampling['episodes'].max() / max(1, sampling['episodes'].min()))
print('saved:', OUT / 'task_sampling_candidates.csv')

## 分布を見る
task別episode数とepisode長の分布を確認します。

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(10, 4))
tasks['episodes'].hist(bins=min(50, max(10, len(tasks)//2)))
plt.xlabel('episodes per task')
plt.ylabel('task count')
plt.title('Task episode-count distribution')
plt.show()
if 'frames' in episodes and episodes['frames'].notna().any():
    plt.figure(figsize=(10, 4))
    episodes['frames'].dropna().hist(bins=50)
    plt.xlabel('frames per episode')
    plt.ylabel('episode count')
    plt.title('Episode length distribution')
    plt.show()

## 次
公開fallbackでpipelineを確認した場合は、次回運営GPUを本来の目的で起動した際に `libero_combined_20hz/meta/` だけを回収し、`PARC_DATASET_ROOT` をそのrootへ向けて正本Inventoryを再生成します。

その後Static Quality Analyzerで movement/path/jerk/idle をepisode単位で追加します。success/collision/replayabilityはraw simulator stateを確保できる場合だけReplay Validatorで追加します。